# Phase 5 (exploratory): training a ternary LM **from scratch** with the counter vs the standard recipe (Colab L4)

**Question.** Ternary LLMs (BitNet b1.58) are trained with **fp32 latent weights + AdamW** and a straight-through
estimator: ~16 bytes of training state per weight. Can the counter (ternary weights + an int8 counter, **~2 bytes**)
train a model from scratch, and how close does it get?

| arm | linear layers | training state / weight |
|---|---|---|
| fp | full precision (reference, not ternary) | 16 B |
| **qat** | ternary, BitNet b1.58 recipe (latent fp32 + AdamW + STE) — **the standard baseline** | 16 B |
| **counter** | ternary, int8 counter, no latent weights | **2 B** |
| sr | ternary, DQT-style stochastic rounding, no memory | 1 B |

**Setup.** LLaMA-style model (8 layers, d 512, SwiGLU, RoPE): **25.7M weights in the linear layers** (ternary in 3 arms)
plus a 25.8M full-precision embedding tied with the output layer (as in BitNet). Data: TinyStories, GPT-2 tokenizer.
Same data order, model, token budget, schedule shape (1% warmup, linear decay to 10%) and 8-bit activations in every
ternary arm. Metric: **validation loss / perplexity** on 1M held-out tokens, plus speed, peak memory and a sample story.

**Protocol (fixed in advance).** Stage 1: each arm's step size is chosen by validation loss after **20M tokens**
(13 short runs, ~1 h). Stage 2: the chosen settings train for **200M tokens** each (~3 h). Single seed (exploratory).

**Reading the result.** counter val loss ≤ qat + 0.10 → competitive at 1/8 of the training state; counter between sr and
qat → the memory helps but costs quality (report the gap); counter ≥ sr → the counter does not help from scratch.

Settings: Runtime → Change runtime type → **L4 GPU**. Everything is saved to `My Drive/ternary_pretrain/`; after a
disconnect, run all again and finished runs are skipped. Keep the tab open.

## Setup (~5 min)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
W = '/content/drive/MyDrive/ternary_pretrain'
os.makedirs(f'{W}/runs', exist_ok=True)
!pip install -q -U transformers datasets
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile /content/ternary_pretrain.py
#!/usr/bin/env python3
"""Pretrain a small LLaMA-style model from scratch with ternary linear layers, comparing update rules.

  fp       full-precision reference (bf16 matmuls, fp32 weights + AdamW): not ternary
  qat      the standard ternary recipe (BitNet b1.58): fp32 latent weights + AdamW, absmean ternarization in the
           forward pass, straight-through estimator; ~16 bytes of training state per weight
  counter  ours: ternary weights + an int8 counter per weight, no latent weights, no optimizer state for them;
           ~2 bytes per weight
  sr       DQT-style stochastic rounding straight onto the ternary grid, no memory; ~1 byte per weight

All arms share the data, model shape, token budget, schedule shape and 8-bit activation quantization (ternary arms).
Embeddings, norms and (ternary arms) one scale per matrix are small fp32 tensors trained with AdamW in every arm.

  python ternary_pretrain.py --arm counter --data /path/tinystories --out runs/counter --tokens 2e8
"""
import argparse, json, math, os, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def get_args():
    p = argparse.ArgumentParser()
    p.add_argument('--arm', required=True, choices=['fp', 'qat', 'counter', 'sr'])
    p.add_argument('--data', required=True, help='folder with train.bin / val.bin (uint16 GPT-2 tokens)')
    p.add_argument('--out', required=True)
    p.add_argument('--d', type=int, default=512)
    p.add_argument('--layers', type=int, default=8)
    p.add_argument('--heads', type=int, default=8)
    p.add_argument('--ffn', type=int, default=1408)
    p.add_argument('--seq', type=int, default=512)
    p.add_argument('--bs', type=int, default=64, help='sequences per optimizer step')
    p.add_argument('--micro', type=int, default=16, help='sequences per forward pass (gradient accumulation; keeps the 50k-vocab logits small)')
    p.add_argument('--tokens', type=float, default=2e8, help='training tokens')
    p.add_argument('--lr', type=float, default=1.5e-3, help='AdamW lr (all weights for fp/qat; small fp tensors otherwise)')
    p.add_argument('--wd', type=float, default=0.1, help='weight decay on matrices (fp/qat)')
    p.add_argument('--warmup', type=float, default=0.01, help='fraction of steps; then linear decay to 10%%')
    p.add_argument('--vote', type=float, default=0.3, help='counter: votes per step at unit normalized gradient')
    p.add_argument('--tau', type=int, default=8, help='counter: flip threshold')
    p.add_argument('--leak', type=float, default=0.0, help='counter: stochastic decay of each counter toward 0')
    p.add_argument('--sr_step', type=float, default=0.01, help='sr: move probability at unit normalized gradient')
    p.add_argument('--clip', type=float, default=1.0)
    p.add_argument('--eval_every', type=float, default=1e7, help='evaluate every N training tokens')
    p.add_argument('--eval_tokens', type=float, default=1e6)
    p.add_argument('--seed', type=int, default=0)
    return p.parse_args()


# ----------------------------------------------------------------------------------------------- layers
def act_quant(x):
    """Per-token absmax int8 activations (BitNet b1.58), straight-through."""
    s = 127.0 / x.detach().abs().amax(-1, keepdim=True).clamp(min=1e-5)
    return x + ((x.detach() * s).round().clamp(-128, 127) / s - x.detach())


class TernaryMatmul(torch.autograd.Function):
    """y = x @ t^T with t in {-1,0,1} (int8 buffer, no gradient). Backward returns dL/dx and leaves dL/dt on the
    layer for its integer update rule; dL/dt is never accumulated across steps."""
    @staticmethod
    @torch.amp.custom_fwd(device_type='cuda')
    def forward(ctx, x, layer):
        ctx.layer = layer
        ctx.save_for_backward(x)
        return F.linear(x, layer.t.to(x.dtype))

    @staticmethod
    @torch.amp.custom_bwd(device_type='cuda')
    def backward(ctx, go):
        (x,) = ctx.saved_tensors
        layer = ctx.layer
        gx = go @ layer.t.to(go.dtype)
        g2, x2 = go.reshape(-1, go.shape[-1]), x.reshape(-1, x.shape[-1])
        gW = (g2.t() @ x2).float()
        layer.gW = gW if layer.gW is None else layer.gW + gW
        return gx, None


class TLinear(nn.Module):
    def __init__(self, din, dout, arm, args):
        super().__init__()
        self.arm, self.args = arm, args
        W = torch.randn(dout, din) * 0.02
        if arm in ('fp', 'qat'):
            self.weight = nn.Parameter(W)
        else:
            beta = W.abs().mean()
            self.register_buffer('t', (W / beta).round().clamp(-1, 1).to(torch.int8))
            self.scale = nn.Parameter(beta.clone())         # one fp32 scale per matrix, trained with AdamW
            if arm == 'counter':
                self.register_buffer('c', torch.zeros(dout, din, dtype=torch.int8))
            self.gW, self.flips = None, 0

    def forward(self, x):
        if self.arm == 'fp':
            return F.linear(x, self.weight)
        x = act_quant(x)
        if self.arm == 'qat':
            W = self.weight
            b = W.detach().abs().mean().clamp(min=1e-5)
            Wq = (W / b).round().clamp(-1, 1) * b
            return F.linear(x, W + (Wq - W).detach())
        return TernaryMatmul.apply(x, self) * self.scale

    @torch.no_grad()
    def update(self, strength):
        """strength: the scheduled vote (counter) or move probability (sr) at unit normalized gradient."""
        g, self.gW = self.gW, None
        if g is None:
            return
        s = -g / g.pow(2).mean().sqrt().clamp(min=1e-30)
        t = self.t.float()
        if self.arm == 'counter':
            a = self.args
            cf = self.c.float()
            if a.leak:
                cf = cf - torch.sign(cf) * (torch.rand_like(cf) < (a.leak * cf.abs()).clamp(max=1)).float()
            cf = (cf + torch.floor(strength * s + torch.rand_like(s))).clamp(-127, 127)
            up, dn = cf >= a.tau, cf <= -a.tau
            nt = torch.where(up, (t + 1).clamp(max=1), torch.where(dn, (t - 1).clamp(min=-1), t))
            cf = torch.where(up | dn, torch.zeros_like(cf), cf)
            self.c.copy_(cf.to(torch.int8))
        else:                                                # sr
            d = strength * s
            move = torch.rand_like(d) < d.abs().clamp(max=1)
            nt = torch.where(move & (d > 0), (t + 1).clamp(max=1), torch.where(move & (d < 0), (t - 1).clamp(min=-1), t))
        self.flips += int((nt != t).sum())
        self.t.copy_(nt.to(torch.int8))


# ----------------------------------------------------------------------------------------------- model
class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))

    def forward(self, x):
        return x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + 1e-6).to(x.dtype) * self.w


def rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    return torch.stack((x1 * cos - x2 * sin, x1 * sin + x2 * cos), -1).flatten(-2)


class Block(nn.Module):
    def __init__(self, a):
        super().__init__()
        d, h, f = a.d, a.heads, a.ffn
        self.h = h
        self.n1, self.n2 = RMSNorm(d), RMSNorm(d)
        self.q, self.k, self.v, self.o = (TLinear(d, d, a.arm, a) for _ in range(4))
        self.gate, self.up, self.down = TLinear(d, f, a.arm, a), TLinear(d, f, a.arm, a), TLinear(f, d, a.arm, a)

    def forward(self, x, cos, sin):
        B, T, D = x.shape
        y = self.n1(x)
        q, k, v = (m(y).view(B, T, self.h, D // self.h).transpose(1, 2) for m in (self.q, self.k, self.v))
        q, k = rope(q, cos, sin), rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D)
        x = x + self.o(y)
        y = self.n2(x)
        return x + self.down(F.silu(self.gate(y)) * self.up(y))


class LM(nn.Module):
    def __init__(self, a, vocab=50304):
        super().__init__()
        self.emb = nn.Embedding(vocab, a.d)
        nn.init.normal_(self.emb.weight, std=0.02)
        self.blocks = nn.ModuleList(Block(a) for _ in range(a.layers))
        self.norm = RMSNorm(a.d)
        hd = a.d // a.heads
        inv = 1.0 / (10000 ** (torch.arange(0, hd, 2).float() / hd))
        ang = torch.outer(torch.arange(a.seq).float(), inv)
        self.register_buffer('cos', ang.cos()[None, None], persistent=False)
        self.register_buffer('sin', ang.sin()[None, None], persistent=False)

    def forward(self, idx, targets=None):
        T = idx.shape[1]
        x = self.emb(idx)
        cos, sin = self.cos[:, :, :T], self.sin[:, :, :T]
        for b in self.blocks:
            x = b(x, cos, sin)
        logits = F.linear(self.norm(x), self.emb.weight)       # tied, full precision (as in BitNet)
        if targets is None:
            return logits
        return F.cross_entropy(logits.float().view(-1, logits.shape[-1]), targets.view(-1))


# ----------------------------------------------------------------------------------------------- training
def main():
    a = get_args()
    os.makedirs(a.out, exist_ok=True)
    torch.manual_seed(a.seed); np.random.seed(a.seed)
    dev = torch.device('cuda')
    train = np.memmap(os.path.join(a.data, 'train.bin'), dtype=np.uint16, mode='r')
    val = np.memmap(os.path.join(a.data, 'val.bin'), dtype=np.uint16, mode='r')
    model = LM(a).to(dev)
    tl = [m for m in model.modules() if isinstance(m, TLinear)]
    n_t = sum(m.t.numel() if hasattr(m, 't') else m.weight.numel() for m in tl)
    n_all = sum(p.numel() for p in model.parameters()) + (sum(m.t.numel() for m in tl if hasattr(m, 't')))
    mats = [p for n, p in model.named_parameters() if p.dim() == 2 and 'emb' not in n]
    rest = [p for n, p in model.named_parameters() if not (p.dim() == 2 and 'emb' not in n)]
    groups = [g for g in ({'params': mats, 'weight_decay': a.wd}, {'params': rest, 'weight_decay': 0.0}) if g['params']]
    opt = torch.optim.AdamW(groups, lr=a.lr, betas=(0.9, 0.95))
    accum = max(1, a.bs // a.micro)
    tok_step = a.bs * a.seq
    steps = int(a.tokens // tok_step)
    warm = max(1, int(a.warmup * steps))
    sched = lambda i: (i + 1) / warm if i < warm else 1.0 - 0.9 * (i - warm) / max(1, steps - warm)
    state_b = {'fp': 16, 'qat': 16, 'counter': 2, 'sr': 1}[a.arm]
    print(f'arm {a.arm} | {n_t / 1e6:.1f}M weights in linear layers ({"ternary" if a.arm != "fp" else "fp"}), '
          f'{n_all / 1e6:.1f}M in total | training state of the linear layers ~{state_b} B/weight = '
          f'{n_t * state_b / 2**20:.0f} MB | {steps} steps of {tok_step} tokens', flush=True)

    rng = np.random.default_rng(a.seed)
    def batch(data, n):
        ix = rng.integers(0, len(data) - a.seq - 1, n)
        x = np.stack([data[i:i + a.seq + 1].astype(np.int64) for i in ix])
        x = torch.from_numpy(x).pin_memory().to(dev, non_blocking=True)
        return x[:, :-1], x[:, 1:]

    n_eval = int(a.eval_tokens // a.seq)
    ev_ix = np.arange(n_eval) * ((len(val) - a.seq - 1) // n_eval)     # fixed, spread over the val split
    @torch.no_grad()
    def evaluate():
        model.eval(); tot = 0.0
        for i in range(0, n_eval, a.micro):
            x = np.stack([val[j:j + a.seq + 1].astype(np.int64) for j in ev_ix[i:i + a.micro]])
            x = torch.from_numpy(x).to(dev)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                tot += model(x[:, :-1], x[:, 1:]).item() * len(x)
        model.train()
        return tot / n_eval

    log = open(os.path.join(a.out, 'log.jsonl'), 'w')
    curve, t0, next_eval = [], time.time(), a.eval_every
    torch.cuda.reset_peak_memory_stats()
    for step in range(steps):
        f = sched(step)
        for g in opt.param_groups:
            g['lr'] = a.lr * f
        for _ in range(accum):                               # counter / sr: dL/dt accumulates on each layer
            x, y = batch(train, a.micro)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                loss = model(x, y)
            (loss / accum).backward()
        if a.clip:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.grad is not None], a.clip)
        opt.step(); opt.zero_grad(set_to_none=True)
        if a.arm in ('counter', 'sr'):
            strength = (a.vote if a.arm == 'counter' else a.sr_step) * f
            for m in tl:
                m.update(strength)
        done = (step + 1) * tok_step
        if step % 50 == 0 or step + 1 == steps:
            el = time.time() - t0
            fl = sum(m.flips for m in tl) / n_t if a.arm in ('counter', 'sr') else None
            print(f'step {step + 1}/{steps} loss {loss.item():.3f} | {done / el / 1e3:.0f}k tok/s | ETA {(steps - step - 1) * el / (step + 1) / 60:.0f} min'
                  + (f' | flips so far {100 * fl:.2f}% of weights' if fl is not None else ''), flush=True)
            log.write(json.dumps(dict(step=step + 1, tokens=done, loss=loss.item(), flips=fl)) + '\n'); log.flush()
        if done >= next_eval or step + 1 == steps:
            vl = evaluate()
            curve.append((done, vl))
            print(f'EVAL tokens {done / 1e6:.0f}M val loss {vl:.4f} ppl {math.exp(vl):.2f}', flush=True)
            log.write(json.dumps(dict(tokens=done, val_loss=vl)) + '\n'); log.flush()
            next_eval += a.eval_every
            if not math.isfinite(vl):
                print('diverged: stopping', flush=True); break
    el = time.time() - t0
    # a short greedy sample from the trained (ternary) model
    model.eval()
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained('gpt2')
    ids = torch.tensor([tok.encode('Once upon a time')], device=dev)
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
        for _ in range(60):
            ids = torch.cat([ids, model(ids[:, -a.seq:])[:, -1:].argmax(-1)], 1)
    sample = tok.decode(ids[0])
    vl = curve[-1][1]
    res = dict(arm=a.arm, args=vars(a), steps=step + 1, tokens=done, val_loss=vl, val_ppl=math.exp(vl) if math.isfinite(vl) else None,
               curve=curve, seconds=el, tok_per_s=done / el, peak_mem_gb=torch.cuda.max_memory_allocated() / 2**30,
               state_bytes_per_weight=state_b, linear_weights=n_t, total_params=n_all,
               flips_frac=(sum(m.flips for m in tl) / n_t if a.arm in ('counter', 'sr') else None), sample=sample)
    print('SAMPLE:', sample.replace('\n', ' '), flush=True)
    print('RESULT', json.dumps({k: v for k, v in res.items() if k not in ('args', 'curve', 'sample')}), flush=True)
    json.dump(res, open(os.path.join(a.out, 'result.json'), 'w'), indent=1)


if __name__ == '__main__':
    main()

## Data (~15 min the first time, then reused from Drive): TinyStories → GPT-2 tokens

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer
DATA = f'{W}/data'
os.makedirs(DATA, exist_ok=True)
tok = AutoTokenizer.from_pretrained('gpt2'); EOS = tok.eos_token_id
for split, name, cap in (('validation', 'val', 5e6), ('train', 'train', 3.2e8)):
    path = f'{DATA}/{name}.bin'
    if os.path.exists(path):
        print(name, 'exists:', os.path.getsize(path) // 2, 'tokens'); continue
    ds = load_dataset('roneneldan/TinyStories', split=split)
    ds = ds.map(lambda b: {'ids': [x + [EOS] for x in tok(b['text'])['input_ids']]}, batched=True,
                num_proc=os.cpu_count(), remove_columns=ds.column_names)
    out, n = open(path + '.tmp', 'wb'), 0
    for i in range(0, len(ds), 20000):
        arr = np.concatenate([np.asarray(x, dtype=np.uint16) for x in ds[i:i + 20000]['ids']])
        out.write(arr.tobytes()); n += len(arr)
        if n >= cap: break
    out.close(); os.rename(path + '.tmp', path)
    print(name, n, 'tokens')

## Quick check (~5 min): every arm trains 2M tokens without errors and its loss goes down

In [ ]:
import subprocess, sys, json, time, glob, math
PY = [sys.executable, '/content/ternary_pretrain.py', '--data', DATA]
def check(arm, extra):
    r = subprocess.run(PY + ['--arm', arm, '--tokens', '2e6', '--eval_every', '1e6', '--eval_tokens', '2e5',
                             '--out', f'/content/check/{arm}'] + extra, capture_output=True, text=True)
    lines = [l for l in r.stdout.splitlines() if l.startswith(('arm ', 'EVAL', 'RESULT', 'SAMPLE')) or ' loss ' in l]
    print(f'--- {arm} ---'); print('\n'.join('  ' + l[:200] for l in lines[:3] + lines[-4:]))
    if r.returncode: print(r.stdout[-2000:]); print(r.stderr[-3000:])
    assert r.returncode == 0, f'{arm}: fix the error above'
for arm, extra in (('fp', ['--lr', '1e-3']), ('qat', ['--lr', '3e-3']), ('counter', ['--vote', '0.3', '--tau', '8']),
                   ('sr', ['--sr_step', '0.01'])):
    check(arm, extra)
print('quick check passed')

## Helpers

In [ ]:
def run_jobs(jobs, every=240):
    todo = [j for j in jobs if not os.path.exists(f'{W}/runs/{j[0]}/result.json')]
    print(f'{len(jobs) - len(todo)} of {len(jobs)} already done', flush=True)
    t0 = time.time()
    for k, (name, a) in enumerate(todo):
        log = open(f'{W}/runs/{name}.out', 'w')
        p = subprocess.Popen(PY + a + ['--out', f'{W}/runs/{name}'], stdout=log, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(every)
            lines = [l for l in open(f'{W}/runs/{name}.out').read().splitlines() if l.startswith(('step', 'EVAL'))]
            if lines: print(f'[{(time.time() - t0) / 60:5.0f} min] {name} ({k + 1}/{len(todo)}): {lines[-1][:150]}', flush=True)
        log.close()
        if p.returncode: print(f'!! {name} failed:\n' + '\n'.join(open(f"{W}/runs/{name}.out").read().splitlines()[-12:]))
    print(f'done in {(time.time() - t0) / 60:.0f} min', flush=True)
def load(n):
    p = f'{W}/runs/{n}/result.json'
    return json.load(open(p)) if os.path.exists(p) else None
GRID = {   # listed from gentlest to most aggressive
    'fp':      [('lr1e-3', ['--lr', '1e-3']), ('lr3e-3', ['--lr', '3e-3'])],
    'qat':     [('lr1.5e-3', ['--lr', '1.5e-3']), ('lr3e-3', ['--lr', '3e-3']), ('lr6e-3', ['--lr', '6e-3'])],
    'counter': [('v0.1_t8', ['--vote', '0.1', '--tau', '8']), ('v0.3_t8', ['--vote', '0.3', '--tau', '8']),
                ('v1_t8', ['--vote', '1.0', '--tau', '8']), ('v1_t4', ['--vote', '1.0', '--tau', '4'])],
    'sr':      [('s0.001', ['--sr_step', '0.001']), ('s0.003', ['--sr_step', '0.003']),
                ('s0.01', ['--sr_step', '0.01']), ('s0.03', ['--sr_step', '0.03'])],
}

## Stage 1 (~1 h): choose each arm's step size on 20M tokens

In [ ]:
SHORT = ['--tokens', '2e7', '--eval_every', '1e7']
run_jobs([(f'tune_{arm}_{tag}', ['--arm', arm] + a + SHORT) for arm, opts in GRID.items() for tag, a in opts])
BEST = {}
print('Stage 1: validation loss after 20M tokens (lower is better)')
for arm, opts in GRID.items():
    sc = [(load(f'tune_{arm}_{tag}'), tag, a) for tag, a in opts]
    sc = [(r['val_loss'], tag, a) for r, tag, a in sc if r and r['val_loss'] is not None and math.isfinite(r['val_loss'])]
    if not sc: print(f'  {arm}: no finished run'); continue
    best = min(sc, key=lambda x: x[0]); BEST[arm] = best
    edge = '  (EDGE of the grid)' if best[1] in (opts[0][0], opts[-1][0]) else ''
    print(f'  {arm:8s} ' + ' | '.join(f'{tag} {v:.3f}' for v, tag, _ in sc) + f'  -> {best[1]}{edge}')

## Stage 2 (~3 h): the chosen settings, 200M tokens each

In [ ]:
MAIN = ['--tokens', '2e8', '--eval_every', '2e7']
ORDER = ['counter', 'qat', 'sr', 'fp']            # the two arms that matter most first
run_jobs([(f'main_{arm}', ['--arm', arm] + BEST[arm][2] + MAIN) for arm in ORDER if arm in BEST], every=600)

## Results

In [ ]:
import matplotlib.pyplot as plt
R = {arm: load(f'main_{arm}') for arm in ORDER}
print('Stage 1 choices:', {a: b[1] for a, b in BEST.items()})
print(f"\n{'arm':8s} {'val loss':>8s} {'ppl':>7s} {'state B/w':>9s} {'peak GB':>8s} {'k tok/s':>8s} {'flips':>8s}")
for arm in ('fp', 'qat', 'counter', 'sr'):
    r = R.get(arm)
    if not r: print(f'{arm:8s} missing'); continue
    fl = f"{100 * r['flips_frac']:.1f}%" if r.get('flips_frac') is not None else '-'
    print(f"{arm:8s} {r['val_loss']:8.4f} {r['val_ppl'] or float('nan'):7.2f} {r['state_bytes_per_weight']:9d} "
          f"{r['peak_mem_gb']:8.1f} {r['tok_per_s'] / 1e3:8.0f} {fl:>8s}")
q, c, s = (R.get(k) for k in ('qat', 'counter', 'sr'))
if q and c:
    gap = c['val_loss'] - q['val_loss']
    verdict = ('competitive (within 0.10 nats of the standard recipe)' if gap <= 0.10 else
               'between SR and the standard recipe' if s and c['val_loss'] < s['val_loss'] else 'no better than SR')
    print(f'\ncounter - qat = {gap:+.3f} nats (ppl x{math.exp(gap):.2f}) -> {verdict}')
if s and q and c:
    print(f"gap closed by the counter between SR and QAT: {100 * (s['val_loss'] - c['val_loss']) / max(1e-9, s['val_loss'] - q['val_loss']):.0f}%")
print('\nnote: peak GB is what this simple implementation uses; the state column is the memory the method needs')
for arm in ('fp', 'qat', 'counter', 'sr'):
    r = R.get(arm)
    if r: print(f"\n[{arm}] {r['sample'][:300]!r}")
fig, ax = plt.subplots(figsize=(8, 4.5))
for arm in ('fp', 'qat', 'counter', 'sr'):
    r = R.get(arm)
    if r and r['curve']:
        x, y = zip(*r['curve']); ax.plot([t / 1e6 for t in x], y, marker='o', ms=3, label=arm)
ax.set_xlabel('training tokens (M)'); ax.set_ylabel('validation loss'); ax.grid(alpha=.3); ax.legend()
ax.set_title('TinyStories, 25.7M-weight linear layers: from-scratch training'); plt.tight_layout(); plt.show()